# Transcript Intelligence analysis

This notebook is a reviewer-friendly guide to the reproducible analytics and Applied AI system. It reads generated outputs rather than duplicating business logic in notebook cells. Run `make all` from the repository root to rebuild the pipeline, hybrid index, agent runs, evaluations, and tests.


## Method

- Call type uses participant domains plus support and escalation title patterns.
- Themes use a curated multi-label keyword taxonomy over titles, summaries, source topics, and key moments.
- Sentiment retains the supplied meeting score and aggregates sentence-level positive, negative, and neutral labels.
- The account risk score prioritizes evidence review; it is not a churn probability.
- The advanced path uses BM25, TF-IDF, and latent semantic retrieval over evidence-preserving transcript chunks.
- A gated agent graph handles policy, classification, retrieval, risk, synthesis, and human-review routing.


In [ ]:
from pathlib import Path
import csv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSIS = ROOT / "analysis"

def read_csv(name):
    with (ANALYSIS / name).open(encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

meetings = read_csv("meeting_analysis.csv")
call_types = read_csv("call_type_summary.csv")
themes = read_csv("theme_summary.csv")
accounts = read_csv("account_risk.csv")
len(meetings), call_types


## Automated evaluation gates

The evaluation suite measures classification accuracy, retrieval recall at three, citation coverage, PII and prompt-injection controls, RBAC denial behavior, bounded risk scores, audit-chain integrity, and deterministic cost. It fails the build when a threshold regresses.


In [ ]:
import json

eval_report = json.loads((ANALYSIS / "advanced" / "eval_report.json").read_text())
eval_report["metrics"], eval_report["gates"], eval_report["passed"]


## Retrieval architecture

The corpus contains 758 evidence chunks. Each result preserves the meeting ID, account domain, timestamp range, redacted text, source hash, combined score, and component scores. Exact vocabulary comes from BM25, phrase overlap comes from TF-IDF, and latent semantic analysis adds topical similarity without a paid model dependency.


In [ ]:
advanced_metrics = json.loads((ANALYSIS / "advanced" / "advanced_metrics.json").read_text())
{
    "evidence_chunks": advanced_metrics["evidence_chunk_count"],
    "retrieval_dimensions": advanced_metrics["retrieval_dimensions"],
    "selected_clusters": advanced_metrics["cluster_validation"]["selected_clusters"],
    "sample_human_review_rate": advanced_metrics["sample_human_review_rate"],
}


## Where agentic behavior earns its cost

The system keeps ingestion, permissions, redaction, scoring, lineage, and audit logging deterministic. Agent orchestration starts when evidence must be gathered, ambiguous labels resolved, multiple tools combined, or a human reviewer engaged. High-risk, low-confidence, policy-flagged, weak-evidence, and failed runs cannot silently complete.


## Core finding: support absorbs the strongest negative signal

Support calls have the lowest average meeting sentiment and the highest negative-utterance share. External relationship calls look much healthier, so a company-wide average would hide the operational pain handled by support.


In [ ]:
sorted(call_types, key=lambda row: float(row["avg_sentiment_score"]))


## Theme examples and evidence

The theme summary includes a high-risk example for every primary theme. Meeting-level output retains dictionary matches and source topics so reviewers can audit why a label was assigned.


In [ ]:
for row in themes:
    print(f'{row["theme"]:30} {row["meeting_count"]:>3}  {row["example_title"]}')


## Account risk triage

This view combines the maximum meeting-level risk with churn signals, technical issues, and feature gaps. Review the underlying transcript evidence before taking action.


In [ ]:
accounts[:10]


## Additional product opportunities

1. Account risk timeline linking sentiment, recurring issues, feature gaps, and renewal conversations.
2. Incident propagation view connecting internal response meetings to support escalations and later commercial risk.
3. Commitment tracker that adds due-date extraction and completion state to the 397 source action items.
4. Voice-of-customer backlog ranked by frequency, account risk, and affected customer segment.


## Limitations

The source appears synthetic, covers only three months, and includes pre-generated summaries and sentiment. The analysis supports prioritization, not causal inference. Production use requires a human-labeled evaluation set, calibrated thresholds, outcome data, and drift monitoring.
